In [1]:
import os
import re
import cv2
import numpy as np
import nrrd
from tqdm import tqdm

In [2]:
# Define the input folder containing JPG images
input_folder = '/home/projects/registration/Data/BFI_registration/37/BFI/37_Datasets_original'


In [3]:
# Function to extract the image number from the filename
def extract_image_number(filename):
    pattern = re.compile(r'_(\d+)_original\.jpg$')
    match = pattern.search(filename)
    if match:
        return int(match.group(1))
    else:
        print(f"Warning: No number found in filename {filename}. Skipping.")
        return None


In [4]:
# List and sort all JPG images in the folder based on the extracted image number
image_files = [f for f in os.listdir(input_folder) if f.endswith(".jpg")]
image_files.sort(key=lambda f: extract_image_number(f) or float('inf'))


In [5]:
# Function to read, downsample by 3, and extract the blue channel from an image
def extract_blue_channel(img_path):
    image = cv2.imread(img_path)
    if image is None:
        raise ValueError(f"Unable to read image {img_path}.")
    
    # Calculate the downsampled size (3x reduction)
    downsampled_height = image.shape[0] // 3
    downsampled_width = image.shape[1] // 3
    
    # Downsample the image by resizing to the smaller dimensions
    resized_image = cv2.resize(image, (downsampled_width, downsampled_height))
    
    # Extract the blue channel (OpenCV loads images in BGR format)
    blue_channel = resized_image[:, :, 0]
    
    # Ensure the blue channel is of dtype uint8
    blue_channel = blue_channel.astype(np.uint8)
    
    return blue_channel

In [6]:
# Function to align centroids of images
def align_centroid(image, reference_image):
    # Compute the centroid of the current image
    moments = cv2.moments(image)
    if moments["m00"] != 0:
        cx = int(moments["m10"] / moments["m00"])
        cy = int(moments["m01"] / moments["m00"])
    else:
        cx, cy = image.shape[1] // 2, image.shape[0] // 2  # Fallback to center
    
    # Compute the centroid of the reference image
    ref_moments = cv2.moments(reference_image)
    if ref_moments["m00"] != 0:
        ref_cx = int(ref_moments["m10"] / ref_moments["m00"])
        ref_cy = int(ref_moments["m01"] / ref_moments["m00"])
    else:
        ref_cx, ref_cy = reference_image.shape[1] // 2, reference_image.shape[0] // 2  # Fallback to center
    
    # Calculate the translation required to align centroids
    dx = ref_cx - cx
    dy = ref_cy - cy
    
    # Translate the image to align centroids
    translation_matrix = np.float32([[1, 0, dx], [0, 1, dy]])
    aligned_image = cv2.warpAffine(image, translation_matrix, (image.shape[1], image.shape[0]))
    
    return aligned_image

In [7]:
# Extract the blue channel of the first image (reference image for centroid alignment)
reference_image_path = os.path.join(input_folder, image_files[0])
reference_blue_channel = extract_blue_channel(reference_image_path)

In [8]:
# Initialize the list for aligned blue channels
aligned_blue_channels = [reference_blue_channel]  # Start with the reference image

# Align all images based on their centroid relative to the reference image
for filename in tqdm(image_files[1:], desc="Processing images", unit="image"):
    img_path = os.path.join(input_folder, filename)
    
    # Extract the blue channel
    blue_channel = extract_blue_channel(img_path)
    
    # Align the current image to the reference image based on centroid
    aligned_image = align_centroid(blue_channel, reference_blue_channel)
    
    # Append the aligned image to the list
    aligned_blue_channels.append(aligned_image)


Processing images: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1835/1835 [01:34<00:00, 19.44image/s]


In [9]:
# Convert the list of aligned images to a NumPy array
aligned_blue_channels_array = np.stack(aligned_blue_channels, axis=0).astype(np.uint8)

# Save the final aligned stack as a .nrrd file
nrrd.write('37_SimpleStacking_witFID_noHW.nrrd', aligned_blue_channels_array)

print(f"Final shape of the aligned image stack: {aligned_blue_channels_array.shape}")
print(f"Data type of the aligned image stack: {aligned_blue_channels_array.dtype}")

Final shape of the aligned image stack: (1836, 692, 1029)
Data type of the aligned image stack: uint8
